# Install acspype

acspype can be installed using pip:

`pip install acspype`

Code documentation for acspype can be found at [https://iantblack.github.io/acspype/](https://iantblack.github.io/acspype/).

## Import Required Packages for this Example

In [1]:
from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.dates import DayLocator, HourLocator
from matplotlib.ticker import MultipleLocator
import numpy as np

from acspype import ACSTSCor
import acspype as acsproc
import acspype as acsqaqc
from acspype import estimate_chl
from acspype import compute_base_uncertainty

from acspype import download_and_load_goldcopy, reformat_ooi_optaa, get_ooi_optaa_cal  # Convenience functions for OOI data. These have not been thoroughly tested outside of this example.

In [2]:
ret_unc = True

# OOI CE02SHBP Example


In this example, we will be using ACS data from the [OOI Oregon Shelf Benthic Experiment Package (CE02SHBP)](https://thredds.dataexplorer.oceanobservatories.org/thredds/catalog/ooigoldcopy/public/CE02SHBP-LJ01D-08-OPTAAD106-streamed-optaa_sample/catalog.html).

This data is a time-series.

The data made available from OOI already has the nearest neighbor salinity, temperature, and pressure assigned to each ACS timestamp. One thing to consider is that the ACS data from the OOI are not time-lag corrected in these datasets. For simplicity, this example does not handle andy time-lag correction or consider the different heights of the sensors on the platform.

In [3]:
acs_url = 'https://thredds.dataexplorer.oceanobservatories.org/thredds/fileServer/ooigoldcopy/public/CE02SHBP-LJ01D-08-OPTAAD106-streamed-optaa_sample/deployment0011_CE02SHBP-LJ01D-08-OPTAAD106-streamed-optaa_sample_20250722T040018.609592-20250723T030329.050494.nc'
acs = download_and_load_goldcopy(thredds_fileserver_url=acs_url,
                                 save_dir = 'ooi_data/')
acs = reformat_ooi_optaa(ds = acs)
dev = get_ooi_optaa_cal(ds = acs) # The get_ooi_optaa_cal function uses the uid in the dataset attributes.
assert np.all(dev.a_wavelength == acs.a_wavelength)  # Will raise an error if the wavelengths do not match.

In [4]:
tscor = ACSTSCor().to_xarray()  ## Load the TSCor coefficients as an xarray dataset.

In [5]:
# TSCor coeffs for absorption.
psi_t_a = tscor.psi_t.sel(wavelength=acs.a_wavelength)
sigma_psi_t_a = tscor.sigma_psi_t.sel(wavelength=acs.a_wavelength)

psi_s_a = tscor.psi_s_a.sel(wavelength=acs.a_wavelength)
sigma_psi_s_a = tscor.sigma_psi_s_a.sel(wavelength=acs.a_wavelength)


# TSCor coeffs for attenuation.
psi_t_c = tscor.psi_t.sel(wavelength=acs.c_wavelength)
sigma_psi_t_c = tscor.sigma_psi_t.sel(wavelength=acs.c_wavelength)

psi_s_c = tscor.psi_s_c.sel(wavelength=acs.c_wavelength)
sigma_psi_s_c = tscor.sigma_psi_s_c.sel(wavelength=acs.c_wavelength)

In [6]:
## Assume a base uncertainty of 0.1 degC for the internal thermistor.
acs['internal_temperature'] = acsproc.compute_internal_temperature(counts = acs.raw_internal_temperature, return_uncertainty = ret_unc, unc = 0.1)

In [ ]:
acs['a_uncorrected'] = (['time','a_wavelength'],acsproc.compute_uncorrected(signal_counts=acs.a_signal,
                                                   reference_counts = acs.a_reference, 
                                                   path_length=dev.path_length, return_uncertainty=ret_unc))

acs['c_uncorrected'] = (['time','c_wavelength'],acsproc.compute_uncorrected(signal_counts=acs.c_signal,
                                                   reference_counts=acs.c_reference, 
                                                   path_length =dev.path_length, return_uncertainty=ret_unc))

In [ ]:
acs['a_m'] = acsproc.compute_measured(uncorrected = acs.a_uncorrected,
                                                    internal_temperature = acs.internal_temperature, 
                                                    offset = dev.a_offset, 
                                                    func_delta_t = dev.func_a_delta_t,
                                                    return_uncertainty=ret_unc)
acs['c_m'] = acsproc.compute_measured(uncorrected = acs.c_uncorrected,
                                                    internal_temperature = acs.internal_temperature, 
                                                    offset = dev.c_offset, 
                                                    func_delta_t=dev.func_c_delta_t,
                                                    return_uncertainty = ret_unc)

In [ ]:
# discontinuity_index = acsproc.find_discontinuity_index(a_wavelength = acs.a_wavelength,
#                                                        c_wavelength = acs.c_wavelength)
# acs['a_m'], acs['a_discontinuity_offset'] = acsproc.discontinuity_correction(measured = acs.a_m_discontinuity,
#                                                                              discontinuity_index=discontinuity_index,
#                                                                              wavelength_dim='a_wavelength',
#                                                                              shift_method = 'halfway')
# acs['c_m'], acs['c_discontinuity_offset'] = acsproc.discontinuity_correction(measured = acs.c_m_discontinuity,
#                                                                              discontinuity_index=discontinuity_index,
#                                                                              wavelength_dim='c_wavelength',
#                                                                              shift_method = 'halfway')


In [ ]:
tcal = dev.tcal  # The reference temperature value in the device file.

temp_unc = compute_base_uncertainty(0.005, 0.0001)
sal_unc = compute_base_uncertainty(0.0005, 0.00005) * 10


acs['a_mts'] = acsproc.ts_correction(measured = acs.a_m, 
                                     temperature = acs.sea_water_temperature,
                                     salinity = acs.sea_water_practical_salinity, 
                                     psi_temperature = psi_t_a, 
                                     psi_salinity= psi_s_a, 
                                     tcal = tcal,
                                     return_uncertainty = ret_unc,
                                     unc_temperature=temp_unc,
                                     unc_salinity=sal_unc,
                                     sigma_psi_salinity=sigma_psi_s_a, sigma_psi_temperature=sigma_psi_t_a)
acs['c_mts'] = acsproc.ts_correction(measured = acs.c_m, 
                                     temperature = acs.sea_water_temperature,
                                     salinity = acs.sea_water_practical_salinity, 
                                     psi_temperature= psi_t_c, 
                                     psi_salinity = psi_s_c, 
                                     tcal = tcal,
                                     return_uncertainty = ret_unc,
                                     unc_temperature = temp_unc, unc_salinity = sal_unc,
                                     sigma_psi_temperature = sigma_psi_t_c, sigma_psi_salinity=sigma_psi_s_c)



In [ ]:
acs['c_mts']

In [ ]:
acs['a_mts'] = acsproc.zero_shift_correction(mts = acs.a_mts)
acs['c_mts'] = acsproc.zero_shift_correction(mts = acs.c_mts)

In [ ]:
from acspype import is_uncertainties_object
is_uncertainties_object(acs.c_mts)

### Step 3.0: Interpolate to Common Wavelengths
In this step, we will linearly interpolate between wavelength bins for the absorption and attenuation bins to create common wavelength bins for the two channels. 1 nm is used as the step size. Common wavelength bins allow for easier computation for applications that require comparison of attenuation to absorption. If not interpolating, an absorption wavelength bin may differ for a corresponding attenuation wavelength bin by several nanometers.

You may note that a RuntimeWarning occurs that describes invalid values. This likely occurs on ACS spectra where NaNs are still present somewhere along the wavelength dimension.

In [ ]:
acs = acsproc.interpolate_common_wavelengths(ds = acs, 
                                             a_wavelength_dim='a_wavelength',
                                             c_wavelength_dim='c_wavelength',
                                             new_wavelength_dim='wavelength',
                                             step=1,
                                             wavelength_range='infer')

### Step 3.1: Apply Scattering Corrections
In this example, we will only apply the baseline and proportional scattering correction, since these do not rely on empirical scale factors. 

#### Slice to the Reference Wavelength
Values beyond the reference wavelength may result in negative values, which can skew some QAQC tests. For simplicity, we will slice the dataset to remove wavelengths beyond the reference wavelength." Traditionally, 715 nm has been used as the reference.

In [ ]:
reference_wavelength = 715

In [ ]:
acs = acs.sel(wavelength = slice(None, reference_wavelength))

In [ ]:
a_mts_715 = acs.a_mts.sel(wavelength = reference_wavelength, method = 'nearest')
c_mts_715 = acs.c_mts.sel(wavelength = reference_wavelength, method = 'nearest')

#### Baseline Method

In [ ]:
acs['a_mts_baseline'] = acsproc.baseline_scattering_correction(a_mts = acs.a_mts, 
                                                               reference_a = a_mts_715) # Baseline Method from Zaneveld et al. 1994
acs['a_mts_baseline'] = acsproc.zero_shift_correction(mts = acs.a_mts_baseline)

print(f"\nVariable Attributes")
print(acs.a_mts_baseline.attrs)

#### Proportional Method

In [ ]:
acs['a_mts_proportional'] = acsproc.proportional_scattering_correction(a_mts = acs.a_mts, 
                                                                       c_mts = acs.c_mts, 
                                                                       reference_a = a_mts_715, 
                                                                       reference_c = c_mts_715) # Proportional Method from Zaneveld et al. 1994
acs['a_mts_proportional'] = acsproc.zero_shift_correction(mts = acs.a_mts_proportional)

print(f"\nVariable Attributes")
print(acs.a_mts_proportional.attrs)

### Step 4.0: Run QAQC Tests

#### Elapsed Time Test

In [ ]:
elasped_time_fail = 20 * 1000
elapsed_time_suspect = 60 * 1000
acs['flag_elapsed_time'] = acsqaqc.elapsed_time_test(acs.elapsed_time, fail_threshold = elasped_time_fail, suspect_threshold = elapsed_time_suspect)

flags, flag_counts = np.unique(acs.flag_elapsed_time.values, return_counts = True)
print('elapsed_time')
for flag in flags:
    print(f"Flag {flag}: {flag_counts[flags.tolist().index(flag)]}")

print(f"\nVariable Attributes")
print(acs.flag_elapsed_time.attrs)

#### Internal Temperature Test

In [ ]:
acs['flag_internal_temperature'] = acsqaqc.internal_temperature_test(internal_temperature=acs.internal_temperature, dev = dev)

flags, flag_counts = np.unique(acs.flag_internal_temperature.values, return_counts = True)
print('internal_temperature')
for flag in flags:
    print(f"Flag {flag}: {flag_counts[flags.tolist().index(flag)]}")
    
    
print(f"\nVariable Attributes")
print(acs.flag_internal_temperature.attrs)

#### Inf/NaN Test

In [ ]:
acs['flag_a_uncorrected_inf_nan'] = acsqaqc.inf_nan_test(uncorrected = acs.a_uncorrected)
acs['flag_c_uncorrected_inf_nan'] = acsqaqc.inf_nan_test(uncorrected = acs.c_uncorrected)

flags, flag_counts = np.unique(acs.flag_a_uncorrected_inf_nan.values, return_counts = True)
print('a_uncorrected')
for flag in flags:
    print(f"Flag {flag}: {flag_counts[flags.tolist().index(flag)]}")

print("")

flags, flag_counts = np.unique(acs.flag_c_uncorrected_inf_nan.values, return_counts = True)
print('c_uncorrected')
for flag in flags:
    print(f"Flag {flag}: {flag_counts[flags.tolist().index(flag)]}")
    
    
print(f"\nVariable Attributes")
print(acs.flag_a_uncorrected_inf_nan.attrs)
print(acs.flag_c_uncorrected_inf_nan.attrs)

#### Gross Range Test

In [ ]:
acs['flag_c_mts_gross_range'] = acsqaqc.gross_range_test(mts = acs.c_mts, 
                                                         sensor_min = -0.005, sensor_max = 10,
                                                         user_min = 0.001, user_max = 8.5)

acs['flag_a_mts_proportional_gross_range'] = acsqaqc.gross_range_test(mts = acs.a_mts_proportional, 
                                                                      sensor_min = -0.005, sensor_max = 10,
                                                                      user_min = 0.001, user_max = 8.5)

print(f"\nVariable Attributes")
print(acs.flag_c_mts_gross_range.attrs)
print(acs.flag_a_mts_proportional_gross_range.attrs)

In [ ]:
acs['blanket_flag_c_mts_gross_range'] = acsqaqc.blanket_gross_range_test(acs.flag_c_mts_gross_range, 
                                                                 wavelength_dim = 'wavelength', 
                                                                 suspect_threshold = 0.10, 
                                                                 fail_threshold = 0.30, 
                                                                 include_suspect_flags = False)

acs['blanket_flag_a_mts_proportional_gross_range'] = acsqaqc.blanket_gross_range_test(acs.flag_a_mts_proportional_gross_range, 
                                                                         wavelength_dim = 'wavelength', 
                                                                         suspect_threshold = 0.10, 
                                                                         fail_threshold = 0.30, 
                                                                         include_suspect_flags = False)



flags, flag_counts = np.unique(acs.blanket_flag_a_mts_proportional_gross_range.values, return_counts = True)
print('Blanket a_mts_proportional Gross Range')
for flag in flags:
    print(f"Flag {flag}: {flag_counts[flags.tolist().index(flag)]}")

print("")


flags, flag_counts = np.unique(acs.blanket_flag_c_mts_gross_range.values, return_counts = True)
print('Blanket c_mts Gross Range')
for flag in flags:
    print(f"Flag {flag}: {flag_counts[flags.tolist().index(flag)]}")
    
print(f"\nVariable Attributes")
print(acs.blanket_flag_c_mts_gross_range.attrs)
print(acs.blanket_flag_a_mts_proportional_gross_range.attrs)

## Step 4.1: Remove Poor Quality Data

Because OOI distributes ACS data parsed into its raw values, we can probably assume that the gap and syntax tests passed.

In [ ]:
pre = len(acs.time.values)
print(f'Total Number of ACS Spectra Before Removal: {pre}')

In [ ]:
acs = acs.where(acs.flag_elapsed_time != 4, drop = True)  # Remove samples with a flag of 4 for the elapsed time test
print(f'Removed {pre - len(acs.time.values)} samples due to failure of elapsed time test.')
next = len(acs.time.values)

acs = acs.where(acs.flag_internal_temperature != 4, drop = True)  # Remove samples with a flag of 4 for the internal temperature test.
print(f'Removed {next - len(acs.time.values)} samples due to failure of internal temperature test.')
next = len(acs.time.values)

acs = acs.where(acs.flag_a_uncorrected_inf_nan != 4, drop = True)  # Remove samples with a flag of 4 for the absorption inf nan test.
acs = acs.where(acs.flag_c_uncorrected_inf_nan != 4, drop = True)  # Remove samples with a flag of 4 for the absorption inf nan test.4
print(f'Removed {next - len(acs.time.values)} samples due to failure of inf nan test for attenuation or absorption.')
next = len(acs.time.values)

acs = acs.where(acs.blanket_flag_a_mts_proportional_gross_range != 4, drop = True)  # Remove samples with a flag of 4 for the absorption gross range test.
acs = acs.where(acs.blanket_flag_c_mts_gross_range != 4, drop = True)  # Remove samples with a flag of 4 for the absorption gross range test.
print(f'Removed {next - len(acs.time.values)} samples due to failure of blanket gross range test for attenuation or absorption.')

In [ ]:
post = len(acs.time.values)
print(f'Total Number of ACS Spectra After Removal: {post}')
print(f'Removed {round((1-post/pre) * 100,2)}% of samples due to failure of QAQC tests.')

## Sidebar: Quick Plot a Spectrum

In [ ]:
spectrum = acs.sel(time = acs.time.values[-51000])
dt = spectrum.time.values
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex = True)

ax[0].set_title('Absorption')
ax[0].axhline(0, color = 'k', linestyle = '--')
ax[0].axvline(reference_wavelength, color = 'k', linestyle = '--')
ax[0].plot(spectrum.wavelength, spectrum.a_mts, label='a_mts')
ax[0].plot(spectrum.wavelength, spectrum.a_mts_baseline, label = 'a_mts_baseline')
ax[0].plot(spectrum.wavelength, spectrum.a_mts_proportional, label = 'a_mts_proportional')
ax[0].legend(loc = 'upper right', ncols = 1)
ax[0].yaxis.set_major_locator(MultipleLocator(0.25))
ax[0].yaxis.set_minor_locator(MultipleLocator(0.05))

ax[1].set_title('Attenuation')
ax[1].plot(spectrum.wavelength, spectrum.c_mts, label = 'c_mts')
ax[1].axvline(reference_wavelength, color = 'k', linestyle = '--')
ax[1].legend(loc = 'upper right', ncols = 1)
ax[1].yaxis.set_major_locator(MultipleLocator(0.05))
ax[1].yaxis.set_minor_locator(MultipleLocator(0.01))

ax[-1].xaxis.set_major_locator(MultipleLocator(50))
ax[-1].xaxis.set_minor_locator(MultipleLocator(5))

ax[0].set_ylabel(r'Absorption ($m^{-1}$)')
ax[1].set_ylabel(r'Attenuation ($m^{-1}$)')
ax[0].set_xlabel('Wavelength (nm)')
ax[1].set_xlabel('Wavelength (nm)')

### Step 6.0: Smooth Data

For simplicity, we will apply a median filter with a centered window width of 13 to attenuation. It may be more appropriate to bin the data by depth, but the OOI does not provide unique profile identifiers, so estimation of a profile would need to be done. That is outside the scope of this example.

In [ ]:
a = acs.a_mts_proportional.rolling({'time': 13}, center = True, min_periods = 1).median(skipna = True)
a = acsproc.zero_shift_correction(mts = a)

a = a.where(a<10, drop = True)  # Remove values greater than 10 m^-1. 

### Step 7.0: Compute Advanced Data Products
In this example we compute chlorophyll-a from absorption line height using the methods from Roesler and Barnard, 2013. 
It should be noted that this algorithm technically calls for the particulate signal, but the OOI does not take the dissolved samples to calculate the particulate signal.

In [ ]:
a650 = a.sel(wavelength = 650, method = 'nearest') 
a676 = a.sel(wavelength = 676, method = 'nearest') 
a715 = a.sel(wavelength = 715, method = 'nearest') 

acs['estimated_chl'] = estimate_chl(a650,a676,a715,0.02) # Using larger alh_coeff.
acs['estimated_chl'] = acs['estimated_chl'].where(acs.estimated_chl > 0, 0) # Values less than 0 are set to 0. Probably from spectra that were not flagged appropriately.
acs['estimated_chl']  = acs.estimated_chl.rolling({'time':4 * 3 + 1}, center = True, min_periods = 1).median(skipna = True)

In [ ]:
print(f"\nVariable Attributes")
print(acs.estimated_chl.attrs)

## Download Fluorometer Data

In [ ]:
fl = download_and_load_goldcopy('https://thredds.dataexplorer.oceanobservatories.org/thredds/fileServer/ooigoldcopy/public/CE02SHSP-SP001-07-FLORTJ000-recovered_cspp-flort_sample/deployment0013_CE02SHSP-SP001-07-FLORTJ000-recovered_cspp-flort_sample_20190624T165545.113000-20190627T023924.529000.nc')
fl = fl.swap_dims({'obs':'time'})
fl = fl[['fluorometric_chlorophyll_a']]
fl = fl.rolling({'time': 2*3 + 1}, center = True, min_periods = 1).median(skipna = True)

# Plot Data



In [ ]:
cmap = matplotlib.colormaps.get_cmap('viridis')

fig, ax = plt.subplots(1,2,figsize = (6,5), constrained_layout = True, sharex = True, sharey = True)

p0 = ax[0].scatter(acs.time, acs.depth, c = acs.estimated_chl, cmap = cmap, s = 10, vmin = 0, vmax = 25)

p1 = ax[1].scatter(fl.time, fl.depth, c = fl.fluorometric_chlorophyll_a, cmap = cmap, s = 10, vmin = 0, vmax = 25)


ax[-1].set_ylim(1,40)
ax[-1].invert_yaxis()

ax[0].set_title('ACS')
ax[1].set_title('ECO Triplet-w')

ax[0].set_ylabel('Depth (m)')

ax[-1].set_xlim(datetime(2019,6,25,6),datetime(2019,6,27,4))
ax[-1].xaxis.set_major_locator(DayLocator(interval = 1))
ax[-1].xaxis.set_minor_locator(HourLocator(interval = 1))

ax[-1].yaxis.set_major_locator(MultipleLocator(5))
ax[-1].yaxis.set_minor_locator(MultipleLocator(1))

fig.colorbar(p1, ax = ax[1], label = r'Chlorophyll-a (${mg}{\cdot}{m^{-3}}$)',shrink = 0.8, pad = 0.001)

plt.savefig('figures/ce02shsp_profiler_acs_triplet.jpg', dpi = 600)

# Export Data

To export data to a new netCDF, you can use Xarray's builtin to_netcdf function. It is important to define the time encoding so that other programming languages can interpret the time units correctly (Python, MATLAB, R, etc.).


<br>

```
from acspype.core import NC_ENCODING
acs.to_netcdf('processed.nc', encoding=NC_ENCODING)
```